# 地理坐标异常与地区冲突

上一课检查邮编前缀能否找到记录。本课筛选代表点的候选坐标，再核对邮编前缀与州是否同时匹配。最后用一组真实记录比较坐标的均值与中位数。


## 1. 从原表读取

保留邮编前导零。客户和卖家各读取邮编前缀和州两个字段。
同一份州代码参考表也用于前面的客户和卖家清洗。


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

data_dir = project_dir.parent / 'archive'
zip_col = 'geolocation_zip_code_prefix'
state_col = 'geolocation_state'
city_col = 'geolocation_city'
key_cols = [zip_col, state_col]
coordinate_cols = ['latitude', 'longitude']

read_options = {
    'dtype': str, 'encoding': 'utf-8-sig',
    'keep_default_na': False, 'na_values': [''],
}
geo_raw = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv', **read_options)
reference_path = project_dir / 'docs' / 'references' / 'ibge_state_codes_v1.json'
state_codes = set(json.loads(reference_path.read_text(encoding='utf-8'))['state_codes'])

def read_address_keys(entity, key_prefix):
    source_cols = [key_prefix + '_zip_code_prefix', key_prefix + '_state']
    address = pd.read_csv(
        data_dir / ('olist_' + entity + '_dataset.csv'),
        usecols=source_cols, **read_options
    ).rename(columns=dict(zip(source_cols, key_cols)))
    address[zip_col] = address[zip_col].str.strip()
    address[state_col] = address[state_col].str.strip().str.upper()
    return address

customer_keys = read_address_keys('customers', 'customer')
seller_keys = read_address_keys('sellers', 'seller')


## 2. 标记异常并建立坐标候选集

对原始字段做整行去重，再解析经纬度。候选记录需要有效的邮编与州、可解析的经纬度，以及项目粗筛矩形内的坐标。

粗筛矩形为纬度 −35 到 6、经度 −75 到 −28。它是参考地理极点选取的宽松项目规则，用于筛出明显偏离的坐标。框内坐标仍需结合州、市等证据判断。 [IBGE 地理极点表](https://anuario.ibge.gov.br/images/aeb/2024/s1/2_pdf/s1t1101.pdf)


In [2]:
geo = geo_raw.drop_duplicates().copy()
geo[zip_col] = geo[zip_col].str.strip()
geo[state_col] = geo[state_col].str.strip().str.upper()
# 先统一大小写与首尾空白，保留拼写和重音差异以便核查。
geo['city_label'] = geo[city_col].str.strip().str.lower().replace('', pd.NA)

geo['latitude'] = pd.to_numeric(geo['geolocation_lat'], errors='coerce')
geo['longitude'] = pd.to_numeric(geo['geolocation_lng'], errors='coerce')
lat = geo['latitude']
lng = geo['longitude']

parsed = lat.notna() & lng.notna()
world_range_ok = lat.between(-90, 90) & lng.between(-180, 180)
inside_screen = lat.between(-35, 6) & lng.between(-75, -28)
key_ok = geo[zip_col].str.fullmatch('[0-9]{5}', na=False) & geo[state_col].isin(state_codes)

geo['flag_missing_or_unparsed_coordinate'] = ~parsed
geo['flag_outside_world_range'] = parsed & ~world_range_ok
geo['flag_outside_brazil_screen'] = world_range_ok & ~inside_screen
geo['flag_invalid_zip_state'] = ~key_ok
geo['coordinate_candidate'] = world_range_ok & inside_screen & key_ok

candidates = geo.loc[geo['coordinate_candidate']]
display(pd.Series({
    '原表行数': len(geo_raw),
    '去重后的行数': len(geo),
    '坐标缺失或解析失败': geo['flag_missing_or_unparsed_coordinate'].sum(),
    '超出全球经纬度范围': geo['flag_outside_world_range'].sum(),
    '超出项目粗筛矩形': geo['flag_outside_brazil_screen'].sum(),
    '邮编或州格式取值异常': geo['flag_invalid_zip_state'].sum(),
    '可用于代表坐标的候选记录': len(candidates),
}))
display(geo.loc[geo['flag_outside_brazil_screen'], key_cols + coordinate_cols].head(10))


原表行数            1000163
去重后的行数           738332
坐标缺失或解析失败             0
超出全球经纬度范围             0
超出项目粗筛矩形             25
邮编或州格式取值异常            0
可用于代表坐标的候选记录     738307
dtype: int64

,geolocation_zip_code_prefix,geolocation_state,latitude,longitude
387565,18243,SP,28.008978,-15.536867
513631,28165,RJ,41.614052,-8.411675
513754,28155,RJ,42.439286,13.820214
514429,28333,RJ,38.381672,-6.328200
516682,28595,RJ,43.684961,-7.411080
538512,29654,ES,29.409252,-98.484121
538557,29654,ES,21.657547,-101.466766
585242,35179,MG,25.995203,-98.078544
585260,35179,MG,25.995245,-98.078533
695377,45936,BA,38.323939,-6.775035


## 3. 检查邮编、州与城市标签

按邮编统计不同州标签数，找出同一前缀跨州的情况。随后按「邮编前缀＋州」分组，查看有多少组仍有多条候选记录。
同组出现多个城市标签时，先记录差异。拼写、重音和来源记录差异都可能造成多个标签。


In [3]:
states_per_zip = geo.groupby(zip_col)[state_col].nunique()
multi_state_zips = states_per_zip.loc[states_per_zip.gt(1)].index
conflict_groups = (
    geo.loc[geo[zip_col].isin(multi_state_zips)]
    .groupby(key_cols, as_index=False)
    .agg(source_rows=('coordinate_candidate', 'size'),
         candidate_rows=('coordinate_candidate', 'sum'))
)
display(conflict_groups)

candidate_groups = candidates.groupby(key_cols).agg(
    candidate_rows=('latitude', 'size'),
    city_labels=('city_label', 'nunique'),
)
display(pd.Series({
    '出现多个州标签的邮编前缀数': len(multi_state_zips),
    '至少有一条候选记录的邮编＋州组合数': len(candidate_groups),
    '仍有多条候选记录的组合数': candidate_groups['candidate_rows'].gt(1).sum(),
    '候选记录中有多个城市标签的组合数': candidate_groups['city_labels'].gt(1).sum(),
}))


,geolocation_zip_code_prefix,geolocation_state,source_rows,candidate_rows
0,02116,RN,1,1
1,02116,SP,10,10
2,04011,AC,1,1
3,04011,SP,69,69
4,21550,AC,1,1
5,21550,RJ,144,144
6,23056,AC,1,1
7,23056,RJ,30,30
8,72915,DF,1,1
9,72915,GO,9,9


出现多个州标签的邮编前缀数            8
至少有一条候选记录的邮编＋州组合数    19019
仍有多条候选记录的组合数         17820
候选记录中有多个城市标签的组合数      8551
dtype: int64

## 4. 区分三种没有候选坐标的情况

先查邮编前缀是否存在，再查「邮编前缀＋州」是否存在，最后查该组合有没有可用的候选记录。
用组合索引判断匹配，客户和卖家各按原表记录计数。四种状态加总应等于输入行数。


In [4]:
source_prefixes = pd.Index(geo[zip_col].dropna().unique())
source_pairs = pd.MultiIndex.from_frame(geo[key_cols].drop_duplicates())
candidate_pairs = candidate_groups.index

def summarize_address_coverage(address):
    address_pairs = pd.MultiIndex.from_frame(address[key_cols])
    prefix_hit = address[zip_col].isin(source_prefixes)
    pair_hit = pd.Series(address_pairs.isin(source_pairs), index=address.index)
    candidate_hit = pd.Series(address_pairs.isin(candidate_pairs), index=address.index)

    counts = pd.Series({
        '地理表缺少邮编前缀': (~prefix_hit).sum(),
        '邮编前缀存在但州组合缺失': (prefix_hit & ~pair_hit).sum(),
        '州组合存在但没有候选坐标': (pair_hit & ~candidate_hit).sum(),
        '匹配到含候选坐标的州组合': candidate_hit.sum(),
    })
    if counts.sum() != len(address):
        raise ValueError('覆盖分类合计与来源记录数不一致')
    return counts

address_coverage = pd.DataFrame({
    '客户记录': summarize_address_coverage(customer_keys),
    '卖家记录': summarize_address_coverage(seller_keys),
})
display(address_coverage)


,客户记录,卖家记录
地理表缺少邮编前缀,278,7
邮编前缀存在但州组合缺失,0,35
州组合存在但没有候选坐标,1,0
匹配到含候选坐标的州组合,99162,3053


## 5. 比较代表坐标的计算方法

取邮编 28155、州 RJ 这一组，先看各条记录的筛选结果，再比较筛选前后分别算出的均值和中位数。纬度约为 −34.59 的点也通过了粗筛，但离同组其余候选点较远。可以观察这个点怎样影响筛选后的均值。
对经纬度分别取中位数可以构造组内近似代表点，后面会把这条规则应用到每个邮编和州的组合。


In [5]:
example_group = geo.loc[geo[zip_col].eq('28155') & geo[state_col].eq('RJ')]
before_screen = example_group[coordinate_cols]
after_screen = example_group.loc[example_group['coordinate_candidate'], coordinate_cols]
display(example_group[key_cols + coordinate_cols + ['coordinate_candidate']])

representative_comparison = pd.DataFrame([
    before_screen.mean(), before_screen.median(),
    after_screen.mean(), after_screen.median(),
], index=['筛选前均值', '筛选前中位数', '筛选后均值', '筛选后中位数'])
display(representative_comparison.round(5))


,geolocation_zip_code_prefix,geolocation_state,latitude,longitude,coordinate_candidate
513643,28155,RJ,-34.586422,-58.732101,True
513678,28155,RJ,-19.936270,-44.012515,True
513702,28155,RJ,-22.734149,-43.379124,True
513754,28155,RJ,42.439286,13.820214,False
513760,28155,RJ,-21.755936,-41.322124,True


,latitude,longitude
筛选前均值,-11.31470,-34.72513
筛选前中位数,-21.75594,-43.37912
筛选后均值,-24.75319,-46.86147
筛选后中位数,-22.24504,-43.69582


## 我的理解

逐格运行并发来第 4 格的覆盖表，然后回答两道题。

1. 为什么一个坐标通过全球经纬度范围检查，仍然可能被本项目的粗筛规则标记？
因为本项目粗筛规则在覆盖巴西地理极点的矩形做粗筛，范围比实际国境更宽
2. 如果一个邮编前缀在地理表中出现，但其中没有卖家所填的州，应该归入覆盖表的哪一行？为什么单靠邮编前缀会把它算成匹配？
会归在邮编前缀存在但州组合缺失，单用邮编前缀时，只要前缀相同就会计作匹配。

下一课将按「邮编前缀＋州」生成唯一的代表坐标表，再核对关联后客户和卖家行数。
